# 01 — Data Exploration

Visualise dataset statistics for UNBC-McMaster, BioVid, and the neonatal pilot cohort.
Run this notebook before training to verify your data layout.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

DATA_ROOT = Path('../data')

## 1. Dataset sizes and label distributions

In [ ]:
from data.datasets import UNBCMcMasterDataset, BioVidDataset, NeonatalDataset

try:
    unbc   = UNBCMcMasterDataset(str(DATA_ROOT / 'unbc_mcmaster'))
    biovid = BioVidDataset(str(DATA_ROOT / 'biovid'), binary=True)
    neo    = NeonatalDataset(str(DATA_ROOT / 'neonatal'))

    print(f'UNBC-McMaster : {len(unbc):,} recordings')
    print(f'BioVid        : {len(biovid):,} recordings')
    print(f'Neonatal pilot: {len(neo):,} recordings ({len(neo.get_subject_ids())} subjects)')
except FileNotFoundError as e:
    print(f'Dataset not found: {e}')
    print('Please follow docs/DATASET_SETUP.md to prepare your data.')

## 2. Neonatal cohort demographics

In [ ]:
try:
    neo = NeonatalDataset(str(DATA_ROOT / 'neonatal'))
    records = neo.records

    labels = [r['label'] for r in records]
    ga     = [r.get('gestational_age', -1) for r in records]
    procs  = [r.get('procedure', 'unknown') for r in records]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Label distribution
    axes[0].bar(['No Pain', 'Pain'], [labels.count(0), labels.count(1)],
                color=['#4CAF50', '#F44336'])
    axes[0].set_title('Label Distribution')
    axes[0].set_ylabel('Count')

    # Gestational age histogram
    valid_ga = [g for g in ga if g > 0]
    axes[1].hist(valid_ga, bins=14, range=(28, 42), color='#2196F3', edgecolor='white')
    axes[1].set_title('Gestational Age Distribution')
    axes[1].set_xlabel('Gestational Age (weeks)')
    axes[1].set_ylabel('Count')

    # Procedure types
    from collections import Counter
    proc_counts = Counter(procs)
    axes[2].bar(proc_counts.keys(), proc_counts.values(), color='#9C27B0')
    axes[2].set_title('Procedure Types')
    axes[2].set_ylabel('Count')
    plt.xticks(rotation=30, ha='right')

    plt.tight_layout()
    plt.savefig('neonatal_demographics.png', dpi=150)
    plt.show()
    print('Saved neonatal_demographics.png')
except Exception as e:
    print(f'Error: {e}')

## 3. Sample multimodal recording

In [ ]:
try:
    neo = NeonatalDataset(str(DATA_ROOT / 'neonatal'))
    sample = neo[0]

    fig = plt.figure(figsize=(15, 5))
    gs  = gridspec.GridSpec(1, 3)

    # Video frame
    ax1 = fig.add_subplot(gs[0])
    frame = sample['video'][0].permute(1, 2, 0).numpy()
    frame = (frame - frame.min()) / (frame.max() - frame.min() + 1e-8)
    ax1.imshow(frame.clip(0, 1))
    ax1.set_title(f"Video frame 0\nLabel={sample['label']}")
    ax1.axis('off')

    # Mel-spectrogram
    ax2 = fig.add_subplot(gs[1])
    mel = sample['audio'].squeeze().numpy()
    ax2.imshow(mel, aspect='auto', origin='lower', cmap='magma')
    ax2.set_title('Mel-spectrogram')
    ax2.set_xlabel('Frame')
    ax2.set_ylabel('Mel bin')

    # Physiological signals
    ax3 = fig.add_subplot(gs[2])
    physio = sample['physio'].numpy()  # (3, L)
    colors = ['#E53935', '#1E88E5', '#43A047']
    labels_p = ['HR', 'SpO2', 'RR']
    for i, (color, label) in enumerate(zip(colors, labels_p)):
        ax3.plot(physio[i], color=color, label=label, linewidth=1.5)
    ax3.set_title('Physiological signals (z-scored)')
    ax3.set_xlabel('Time (s)')
    ax3.legend()

    plt.tight_layout()
    plt.savefig('sample_multimodal.png', dpi=150)
    plt.show()
    print(f"Subject: {sample['subject']} | GA: {sample.get('gestational_age','?')}w")
except Exception as e:
    print(f'Error: {e}')